# Urban Mobility Optimization: Strategic Hub Localization
### Case Study: Fleet Positioning in Buenos Aires (CABA)

1. **Objective:** Identify strategic points for preventive driver positioning in Buenos Aires.
2. **Methodology:** Use the official taxi stops dataset (as a proxy for historical demand) and apply a **K-Means Clustering** algorithm to solve a **Location-Allocation** problem (Operations Research).

In [14]:
import pandas as pd
import re
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import folium
import networkx as nx
from branca.element import Template
import warnings
import networkx as nx
from scipy.spatial.distance import cdist
warnings.filterwarnings('ignore')

## 1. Data Preparation (ETL)
We extract geographical coordinates from the `geometry` column to enable vector operations with the city's data points.

In [15]:
# Load the Buenos Aires taxi stops dataset
df = pd.read_csv('datasets/paradas_taxis.csv')

def extract_coords(geometry):
    # Original format is POINT (longitude latitude)
    coords = re.findall(r"[-+]?\d*\.\d+|\d+", geometry)
    return float(coords[1]), float(coords[0])

# Generate clean Latitude and Longitude columns
df[['lat', 'lon']] = df['geometry'].apply(lambda x: pd.Series(extract_coords(x)))

print(f"Total analyzed points: {len(df)}")
df[['barrio', 'lat', 'lon']].head()

Total analyzed points: 333


,barrio,lat,lon
0,Flores,-34.627642,-58.463176
1,Villa Crespo,-34.602255,-58.432643
2,Villa Crespo,-34.600265,-58.447319
3,Flores,-34.631550,-58.469374
4,San Nicolas,-34.603021,-58.369557


## 2. Operations Research Modeling
We use **K-Means** to identify 10 "gravity centers" of existing supply.
This analysis minimizes the total distance between any taxi stop (demand) and the nearest Expansion Hub (supply), optimizing the projected **ETA (Estimated Time of Arrival)**.

In [16]:
# Define 10 strategic clusters
K_HUBS = 10
kmeans = KMeans(n_clusters=K_HUBS, random_state=42, n_init=10)

# Assign each stop to a cluster and calculate centroids (Hubs)
df['cluster'] = kmeans.fit_predict(df[['lat', 'lon']])
hubs = kmeans.cluster_centers_

print("Strategic Hub coordinates successfully generated.")

Strategic Hub coordinates successfully generated.


## 3. Interactive Geographical Visualization
We generate a map using the **CartoDB Positron** layer for better urban readability.
- <span style="color: skyblue;">**Blue points**</span> represent existing transport infrastructure (Market Demand).
- <span style="color: red;">**Red markers**</span> represent suggested Expansion Hubs (Supply Centers).

In [17]:
import folium
from branca.element import Element

# 1. Create the base map (Centered in Buenos Aires)
mapa_caba = folium.Map(location=[-34.6037, -58.3816], zoom_start=12, tiles='CartoDB positron')

# 2. Add current taxi stops (Blue Points - Demand)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='#3498db',
        fill=True,
        fill_color='#3498db',
        fill_opacity=0.7,
        popup=f"District: {row['barrio']}"
    ).add_to(mapa_caba)

# 3. Add Expansion Hubs (Red Markers - Supply)
for i, hub in enumerate(hubs):
    folium.Marker(
        location=[hub[0], hub[1]],
        icon=folium.Icon(color='red', icon='car', prefix='fa'), 
        popup=f'Expansion Hub {i+1}'
    ).add_to(mapa_caba)

# 4. Add Strategic Leyend
legend_html = """
<div style="
    position: fixed; 
    bottom: 30px; right: 30px; width: 250px; height: auto; 
    background-color: white; border: 1px solid #d1d1d1; z-index: 9999; 
    font-family: 'Arial', sans-serif; font-size: 13px; color: #333;
    box-shadow: 2px 2px 10px rgba(0,0,0,0.1); border-radius: 8px; 
    padding: 15px; line-height: 1.6;
">
    <b style="font-size: 14px; color: #000;">Strategic Roadmap</b><br>
    <hr style="margin: 8px 0; border: 0; border-top: 1px solid #eee;">
    <div style="display: flex; align-items: center; margin-bottom: 5px;">
        <span style="background: #3498db; border-radius: 50%; width: 12px; height: 12px; display: inline-block; margin-right: 10px;"></span>
        Taxi Stops (Market Demand)
    </div>
    <div style="display: flex; align-items: center;">
        <span style="color: #e74c3c; font-size: 14px; margin-right: 8px;">
            <i class="fa fa-car"></i>
        </span>
        Expansion Hubs (Supply Centers)
    </div>
    <div style="margin-top: 10px; font-size: 11px; color: #777; font-style: italic;">
        Optimized via K-Means Clustering
    </div>
</div>
"""

mapa_caba.get_root().html.add_child(Element(legend_html))

mapa_caba.save('strategic_expansion_map.html')
mapa_caba

## Strategic Conclusions: Dynamic Network Systems

This analysis provides a framework applicable to any urban logistics system:

1. **From Points to Networks:** Efficiency does not depend on isolated assets but on **network capillarity**. Positioning drivers at optimized centroids ensures the lowest possible response time.
2. **Operational Resilience:** Modeling the city this way allows for better fleet rebalancing. If one sector is overwhelmed, the model identifies the nearest hub to redistribute the load.
3. **Mathematical Scalability:** This model is territory-agnostic. The combination of **Unsupervised Learning (K-Means)** and **Spatial Analysis** provides a competitive advantage in international expansion, ensuring decisions are driven by the city's mathematical structure rather than intuition.

## 4. Network Topology & Connectivity Analysis
Beyond spatial clustering, we model the expansion hubs as a **Graph**. This allows us to analyze the "inter-connectivity" of the city. 

- **Nodes:** Expansion Hubs (Strategic Centers).
- **Edges:** Feasible transition paths for drivers between hubs (set at a ~3km threshold).
- **Goal:** Identify the "backbone" of the mobility network and ensure fleet rebalancing is operationally viable.

In [18]:
# Initialize a Graph to represent the 10 Strategic Hubs
G = nx.Graph()

# Add hubs as nodes with their geographical positions
for i, hub in enumerate(hubs):
    G.add_node(i, pos=(hub[1], hub[0])) # (longitude, latitude)

# Calculate Euclidean distance matrix between all hubs
dist_matrix = cdist(hubs, hubs, metric='euclidean')

# Establish connections (edges) based on a proximity threshold
# This simulates the ability of a driver to "jump" or rebalance between nearby hubs
THRESHOLD = 0.05 # Approximate degree threshold (~5km)

for i in range(len(hubs)):
    for j in range(i + 1, len(hubs)):
        if dist_matrix[i, j] < THRESHOLD:
            G.add_edge(i, j, weight=dist_matrix[i, j])


In [19]:
mapa_caba = folium.Map(location=[-34.6037, -58.3816], zoom_start=12, tiles='CartoDB positron')

# We add Network Connectivity 
for u, v in G.edges():
    pos_u = [hubs[u][0], hubs[u][1]]
    pos_v = [hubs[v][0], hubs[v][1]]
    
    folium.PolyLine(
        locations=[pos_u, pos_v],
        color='#2c3e50',
        weight=2,
        opacity=0.6,
        tooltip="Supply Corridor"
    ).add_to(mapa_caba)

# We add current taxi stops (Blue Points - Demand)
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='#3498db',
        fill=True,
        fill_color='#3498db',
        fill_opacity=0.4,
        popup=f"District: {row['barrio']}"
    ).add_to(mapa_caba)

# We add Expansion Hubs (Red Markers - Supply)
for i, hub in enumerate(hubs):
    folium.Marker(
        location=[hub[0], hub[1]],
        icon=folium.Icon(color='red', icon='car', prefix='fa'), 
        popup=f'Expansion Hub {i+1}'
    ).add_to(mapa_caba)

# Strategic Legend
legend_html = """
<div style="
    position: fixed; 
    bottom: 30px; right: 30px; width: 260px; height: auto; 
    background-color: white; border: 1px solid #d1d1d1; z-index: 9999; 
    font-family: 'Arial', sans-serif; font-size: 13px; color: #333;
    box-shadow: 2px 2px 10px rgba(0,0,0,0.1); border-radius: 8px; 
    padding: 15px; line-height: 1.6;
">
    <b style="font-size: 14px; color: #000;">Strategic Roadmap</b><br>
    <hr style="margin: 8px 0; border: 0; border-top: 1px solid #eee;">
    <div style="display: flex; align-items: center; margin-bottom: 5px;">
        <span style="background: #3498db; border-radius: 50%; width: 12px; height: 12px; display: inline-block; margin-right: 10px;"></span>
        Taxi Stops (Market Demand)
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 5px;">
        <span style="color: #e74c3c; font-size: 14px; margin-right: 8px;">
            <i class="fa fa-car"></i>
        </span>
        Expansion Hubs (Supply Centers)
    </div>
    <div style="display: flex; align-items: center;">
        <span style="background: #2c3e50; width: 20px; height: 2px; display: inline-block; margin-right: 10px;"></span>
        Supply Network (Connectivity)
    </div>
    <div style="margin-top: 10px; font-size: 11px; color: #777; font-style: italic;">
        Optimized via K-Means & Graph Theory
    </div>
</div>
"""

mapa_caba.get_root().html.add_child(Element(legend_html))

mapa_caba.save('strategic_expansion_map.html')
mapa_caba

## Network Topology Conclusions & Observations

The transition from a simple spatial distribution (Clustering) to a **Connected Network (Graph Theory)** provides deeper operational insights for fleet management:

1. **Network Resilience & Rebalancing:** The "backbone" of the network identifies where drivers can most efficiently move between high-demand zones. Hubs with high connectivity allow for seamless fleet rebalancing, ensuring that if one area experiences a sudden spike in demand, drivers can redistribute from neighboring hubs with minimal "dead mileage."

2. **The "Isolated Hub" Phenomenon:**
   During our analysis, we identified an **Isolated Node** (a hub with no edges). Within our proximity threshold (approx. 5km), this node remains disconnected from the main urban network.
   - **Strategic Interpretation:** This hub likely serves a **Satellite Zone** (We would have the same behavior with more distant neighborhoods like Mataderos or Lugano).
   - **Business Decision:** While central hubs can share a pool of drivers, an isolated hub requires an **Independent Fleet Strategy**. Drivers positioned here should focus on local "round-trip" demand rather than attempting to rebalance toward the city center, which would incur high operational costs.

3. **Data-Driven Scalability:**
   By combining **K-Means** (Local Optimization) with **Graph Theory** (Global Connectivity), we have built a scalable framework. This allows an International Expansion team to distinguish between "Connected Urban Corridors" and "Strategic Satellite Outposts" in any new market, optimizing both driver satisfaction and platform reliability from Day 1.